In [1]:
import polars as pl

In [ ]:
df = pl.read_parquet("../lichess_eval_parquet_zobr_sorted/part-00000.parquet")
print(df.head(10))

: 

In [3]:
fens = df["zobr64"][:20].to_list()
if fens == sorted(fens):
    print("First 20 are sorted.")
else:
    print("First 20 are NOT sorted.")

First 20 are sorted.


In [4]:
first_row = df.row(0)
print(first_row)

('4r1k1/pb3ppp/2pb4/8/2B5/PP6/1B3PPP/3R1K2 b - -', [{'knodes': 2265, 'depth': 25, 'pvs': [{'cp': 61, 'mate': None, 'line': 'e8d8 b3b4 b7c8 b2c1 d6e7 d1d8 e7d8 c1e3 a7a5 e3c5'}]}], 3765241343, b'\xc4;h?\xcb\xa7\xd1J\x00\x00\x00\x00\xe0m\x05\xff')


In [5]:
result = (
    df.with_row_index("idx")
    .group_by("zobr64")
    .agg(
        pl.len().alias("count"),
        pl.col("idx").min().alias("first_idx"),
    )
    .filter(pl.col("count") > 1)
    .sort("first_idx")
    .head(1)
)
result

zobr64,count,first_idx
u64,u32,u32
19033401819165,2,364


In [6]:
filtered_rows = df.filter(pl.col("zobr64") == 19033401819165)
print(filtered_rows)

shape: (2, 4)
┌───────────────────────────┬──────────────────────────┬────────────────┬──────────────────────────┐
│ fen                       ┆ evals                    ┆ zobr64         ┆ zobr128                  │
│ ---                       ┆ ---                      ┆ ---            ┆ ---                      │
│ str                       ┆ list[struct[3]]          ┆ u64            ┆ binary                   │
╞═══════════════════════════╪══════════════════════════╪════════════════╪══════════════════════════╡
│ r1b2rk1/ppppqppp/2n2n2/b7 ┆ [{4539,23,[{-49,null,"c3 ┆ 19033401819165 ┆ b"\xa1\x7f\xf5)H\x10v\xe │
│ /2B1P…                    ┆ d5 f6d…                  ┆                ┆ a\x00\…                  │
│ r1b2rk1/ppppqppp/2n2n2/b7 ┆ [{33869,25,[{-32,null,"c ┆ 19033401819165 ┆ b"\xa1\x7f\xf5)H\x10v\xe │
│ /2B1P…                    ┆ 3d5 f6…                  ┆                ┆ a\x00\…                  │
└───────────────────────────┴──────────────────────────┴────────────────┴────

In [7]:
filtered_rows["evals"]

evals
list[struct[3]]
"[{4539,23,[{-49,null,""c3d5 f6d5 e4d5 c6e5 f3e5 e7e5 c1b2 e5g5 b3a3 a5d2""}, {-54,null,""c1b2 a5c3 b2c3 d7d6 a1e1 f6d7 b3b2 d7e5 f3e5 d6e5""}, {-54,null,""c1b2 a5c3 b2c3 d7d6 a1e1 f6d7 b3b2 d7e5 f3e5 d6e5""}]}, {20176,19,[{-35,null,""c1b2 a5c3 b2c3 d7d6 a1e1 f6d7 c3b2 c6e5 f3e5 d7e5""}, {-47,null,""c3d5 f6d5 e4d5 c6e5 f3e5 e7e5 c1b2 e5g5 d5d6 b7b5""}, … {-103,null,""a1b1 d7d6 c1b2 f6e4 c3d5 e7d8 b2a1 g8h8 b3c2 e4g5""}]}]"
"[{33869,25,[{-32,null,""c3d5 f6d5 e4d5 c6e5 f3e5 e7e5 c1b2 e5g5 b2c1 g5h5""}, {-37,null,""c1b2 a5c3 b2c3 d7d6 a1e1 f6d7 c3b2 d7e5 f3e5 c6e5""}, … {-79,null,""c1a3 d7d6 a1e1 f6d7 c3d5 e7d8 e1d1 g8h8 b3c2 a5b6""}]}]"


In [9]:
filtered_rows["fen"].to_list()

['r1b2rk1/ppppqppp/2n2n2/b7/2B1P3/1QN2N1P/P4PP1/R1B2RK1 w - -',
 'r1b2rk1/ppppqppp/2n2n2/b7/2B1P3/1QN2N1P/P4PP1/R1B2RK1 w Q -']

In [10]:
result = (
    df.with_row_index("idx")
    .group_by("zobr64")
    .agg(
        pl.len().alias("count"),
        pl.col("idx").min().alias("first_idx"),
    )
    .filter(pl.col("count") > 1)
    .sort("first_idx")
    .head(100)
)
result

zobr64,count,first_idx
u64,u32,u32
19033401819165,2,364
45162543674492,2,855
80935640104210,2,1542
105027789383870,2,1960
127565082671948,2,2384
…,…,…
2913776575077919,2,55590
2984773519444815,2,56997
2991571245539467,2,57136


In [14]:
df_fil = df.filter(pl.col("zobr64") == 45162543674492)
df_fil["fen"].to_list()

['r2q1rk1/pppn1pbp/2np2p1/4p3/2PP4/2N1PBPP/PP3P2/R1BQ1RK1 w - -',
 'r2q1rk1/pppn1pbp/2np2p1/4p3/2PP4/2N1PBPP/PP3P2/R1BQ1RK1 w Qq -']

In [17]:
fen_longest_depth = (
    df_fil.with_columns(
        pl.col("evals")
        .list.eval(pl.element().struct.field("depth"))
        .list.max()
        .alias("row_longest_depth")
    )
    .group_by("fen")
    .agg(pl.col("row_longest_depth").max().alias("longest_depth"))
    .sort("longest_depth", descending=True)
)

fen_longest_depth.head(10)

fen,longest_depth
str,i64
"""r2q1rk1/pppn1pbp/2np2p1/4p3/2P…",36
"""r2q1rk1/pppn1pbp/2np2p1/4p3/2P…",32


In [19]:
fen_longest_depth["fen"].to_list()

['r2q1rk1/pppn1pbp/2np2p1/4p3/2PP4/2N1PBPP/PP3P2/R1BQ1RK1 w - -',
 'r2q1rk1/pppn1pbp/2np2p1/4p3/2PP4/2N1PBPP/PP3P2/R1BQ1RK1 w Qq -']

In [22]:
df_dup = (
    df.with_row_index("idx")
    .group_by("zobr64")
    .agg(
        pl.len().alias("count"),
        pl.col("idx").min().alias("first_idx"),
    )
    .filter(pl.col("count") > 1)
    .sort("first_idx")
)
df_dup

zobr64,count,first_idx
u64,u32,u32
19033401819165,2,364
45162543674492,2,855
80935640104210,2,1542
105027789383870,2,1960
127565082671948,2,2384
…,…,…
109212395753314864,2,2096818
109251362333487728,2,2097588
109263363518130074,2,2097809


In [24]:
zobr64_dup = df_dup["zobr64"]
zobr64_dup

zobr64
u64
19033401819165
45162543674492
80935640104210
105027789383870
127565082671948
…
109212395753314864
109251362333487728
109263363518130074


In [26]:
zobr64_dup_fen_diff = (
    df.join(
        df_dup.select("zobr64"),
        on="zobr64",
        how="semi",
    )
    .with_columns(
        pl.col("fen").str.split(" ").list.get(0).alias("fen_board"),
        pl.col("fen").str.split(" ").list.get(1).alias("fen_turn"),
        pl.col("fen").str.split(" ").list.get(2).alias("fen_castling"),
        pl.col("fen").str.split(" ").list.get(3).alias("fen_ep"),
    )
    .group_by("zobr64")
    .agg(
        pl.len().alias("rows"),
        pl.col("fen").n_unique().alias("fen_unique"),
        pl.col("fen_board").n_unique().alias("board_unique"),
        pl.col("fen_turn").n_unique().alias("turn_unique"),
        pl.col("fen_castling").n_unique().alias("castling_unique"),
        pl.col("fen_ep").n_unique().alias("ep_unique"),
        pl.col("fen").sort().alias("fens"),
    )
    .with_columns(
        (
            (pl.col("board_unique") == 1)
            & (pl.col("turn_unique") == 1)
            & (pl.col("ep_unique") == 1)
            & (pl.col("castling_unique") > 1)
        ).alias("castle_rights_only")
    )
    .with_columns(
        pl.when(pl.col("castle_rights_only"))
        .then(pl.lit("castle_rights_only"))
        .otherwise(pl.lit("other_fen_difference"))
        .alias("difference_type")
    )
    .sort("zobr64")
)

zobr64_dup_fen_diff.group_by("difference_type").len()


difference_type,len
str,u32
"""castle_rights_only""",3320
